# Auditing a regression model with `c4fairness` (student grades)

A regression model that looks good on paper can still fail people unevenly. Report a single RMSE
and you have compressed every mistake the model makes into one number — which is exactly the
number that hides *where* and *for whom* it goes wrong. Two models with the same RMSE can behave
completely differently: one spreads its error evenly, the other is near-perfect for most students
and badly biased for a subgroup. Aggregate accuracy cannot tell them apart.

The usual next step is to slice by a protected attribute — grade error by sex, by parental
education — and compare. That helps, but it only ever checks the attributes you thought to name,
one at a time, and it averages away any structure *inside* each group. The subgroup that a model
actually fails is often an **intersection** (older students from a particular background taking a
specific kind of course) that no single group-by will isolate.

`c4fairness` takes a different angle. It **clusters the test set on the features** — letting the
data's own geometry define the subgroups — and then reports the model's error *per cluster*,
alongside which protected groups each cluster concentrates. Pockets of high or biased error that
a per-attribute average washes out show up as their own clusters, each with a significance test
attached.

For regression the error is the **signed residual** `y_true - y_pred`, which carries two things
at once:

- its **sign** = the direction of bias (does the model systematically over- or under-predict
  here?),
- its **magnitude** = reliability (how large are the misses, regardless of direction?).

This notebook runs the full audit with the Python API, then shows the two variants that matter in
practice — **Gower distance** for mixed numeric/categorical data and **silhouette-based k-search**
— which the CLI `--experiment` mode sweeps automatically.

Data: `Data/student_performance.csv` (670 students, each row carrying the model's `y_pred`).
Sensitive attributes cover all three kinds: **binary** `sex_F`, **multi-categorical** `Medu`
(mother's education, 0–4), and **numeric** `age`.

In [ ]:
import numpy as np
import pandas as pd
from c4fairness.result_viz import plot_cluster_recap_heatmap
from IPython.display import Image

df = pd.read_csv("../Data/student_performance.csv")
df[["G1", "G2", "studytime", "absences", "sex_F", "Medu", "age", "y_true", "y_pred"]].head()

## 1. The residual, and why its sign matters

`residual = y_true - y_pred`. Positive means the true grade was higher than predicted — the model
**under**-predicted; negative means it **over**-predicted.

Two summaries capture the two failure modes:

- the **mean residual** measures *bias* — a systematic lean in one direction. A well-calibrated
  model has a mean residual near zero overall.
- the **mean absolute residual** measures *typical error size* — how far off predictions are,
  ignoring direction.

Here is the catch that motivates the whole notebook: a mean residual near zero *overall* proves
nothing about fairness. A cluster that the model over-predicts by +2 and another it under-predicts
by −2 cancel perfectly in the global average, while both are badly biased. The aggregate is
blind to exactly the disparity we care about — which is why we go looking cluster by cluster.

In [ ]:
residual = df["y_true"] - df["y_pred"]
print(f"overall mean residual   = {residual.mean():+.3f}   (bias direction)")
print(f"overall mean |residual| = {residual.abs().mean():.3f}   (typical error size)")

## 2. The aggregate view — the lens clustering improves on

Before clustering, look at the conventional fairness check: one number per protected group, the
mean residual within each level of `sex_F` and of `Medu`. This is the standard slice-and-compare
audit.

It is useful but limited in two specific ways, worth naming because they are exactly what
clustering addresses:

1. **It only sees the axis you chose.** Grouping by `Medu` can only reveal disparity that lines
   up with maternal education. If the model actually fails, say, older students in short courses,
   no single group-by will point at them.
2. **It averages within the group.** A `Medu` level whose mean residual is ~0 may still contain a
   sharply biased sub-population, hidden by the students it cancels against.

Read the numbers below as the baseline: a flat table that clustering will turn into something with
internal structure.

In [ ]:
tmp = df.assign(residual=residual)
print("mean residual by sex_F (0=male, 1=female):")
print(tmp.groupby("sex_F")["residual"].mean().round(3), "\n")
print("mean residual by Medu (mother's education 0-4):")
print(tmp.groupby("Medu")["residual"].mean().round(3))

## 3. Encode + cluster (the Euclidean baseline)

Clustering needs a numeric feature matrix and a distance. The most common setup — the one to
understand first before reaching for anything fancier — is **one-hot encoding + Euclidean
distance**, which is what `encode_categoricals(..., distance="euclidean")` prepares.

Two modelling choices worth being deliberate about:

- **`Medu` is marked categorical.** It is an integer *code* (0–4), not a quantity. Left as a
  number, Euclidean distance would assume `Medu=4` is "twice as far" from `0` as `Medu=2` is —
  an ordering the codes do not earn. One-hot encoding turns each level into its own 0/1 column so
  no artificial spacing is imposed.
- **`age` stays continuous.** Here the numeric spacing *is* meaningful (a 2-year gap is genuinely
  smaller than a 10-year gap), so we keep it as one column.

`encode_categoricals` also returns `multiclass_dummies` (mcd) — the map from `Medu` to the dummy
columns it exploded into — which we use later to rebuild a single readable `Medu` column for the
results table. (One subtlety it handles for you: the 0/1 dummy columns are kept out of
`StandardScaler`. Standardising a binary indicator rescales it by its own frequency, which
silently distorts distances — a rare category would blow up into a high-variance axis. More on
that failure mode in §7.)

In [ ]:
from c4fairness.preprocessing import encode_categoricals
from c4fairness.clustering import cluster

sensitive = ["sex_F", "Medu", "age"]
col_lists = {"regular": ["G1", "G2", "studytime", "absences"], "sensitive": sensitive,
             "proxy": [], "special": []}
orig_sensitive = list(sensitive)

dfe, cl, cat_names, mcd, ohe = encode_categoricals(
    df.copy(), col_lists, ["Medu"], "kmeans", distance="euclidean"
)
clustering_cols = cl["regular"] + cl["sensitive"]
res = cluster(dfe[clustering_cols], algorithm="kmeans", distance="euclidean",
              n_clusters=3, random_state=42)
print(f"k = {res.n_clusters}   silhouette = {res.silhouette:.3f}   sizes = {res.cluster_sizes}")

## 4. The per-cluster recap

`make_recap` collapses the clustered data into **one row per cluster**. Passing
`error_type="regression"` tells it to treat the error column as a continuous quantity rather than
a 0/1 outcome, which changes two things:

- it reports the **signed median residual** (`error_mean`, the bias) and its **magnitude**
  (`abs_error_mean`, the reliability), instead of a rate;
- it tests significance with **Mann-Whitney / ANOVA** — location tests appropriate for a
  continuous variable — rather than Fisher's exact test, which is for 2×2 count tables and only
  makes sense for a binary error.

Sensitive features are summarised in **salient** form: one readable column per attribute showing
its dominant category (`Medu_cat`) — except `age`, declared continuous, which is summarised by its
median. Two helper steps make that readable output: `_build_sensitive_analysis_list` names the
columns to analyse, and `apply_salient_reconstruction` rebuilds the human-readable `Medu` from its
dummies.

In [ ]:
from c4fairness.cli import _build_sensitive_analysis_list, apply_salient_reconstruction
from c4fairness.experiments import make_recap

analysis = _build_sensitive_analysis_list(cl["sensitive"], mcd, orig_sensitive, option="salient")
dfe["residual"] = residual.values
res_df = dfe.copy()
res_df["clusters"] = res.labels
apply_salient_reconstruction(res_df, mcd, orig_sensitive)

recap = make_recap(res_df, clustering_cols, sensitive_cols=analysis,
                   error_col="residual", error_type="regression",
                   feature_matrix=res.feature_matrix, continuous_sensitive_cols=["age"])
recap.round(3)

## 5. Reading the recap

Each row is a cluster. The columns you actually audit on:

- **`error_mean`** — the cluster's median signed residual. The **sign** is the story: positive =
  the model under-predicts these students' grades, negative = it over-predicts. Magnitude = how
  strong that lean is.
- **`abs_error_mean`** — typical error size in the cluster. High here even with `error_mean` near
  zero means the model is *unreliable* (big misses in both directions), which is a different
  problem from *biased*.
- **`error_gap`** — this cluster's error minus the error of everyone else (one-vs-all). It answers
  "is this cluster worse than the rest, and by how much?"
- **`error_gap_sig`** — the p-value for that gap (Mann-Whitney). Small = the gap is unlikely to be
  noise. A big gap with a large p-value in a tiny cluster is a warning to not over-read it.
- **`Medu_cat` / `sex_F_value` / `age_value`** — the composition: which maternal-education level
  dominates, the share of female students, the median age. This is *who* the cluster is.

The cluster to worry about is one with a large **absolute** bias *and* a significant gap: that is
a subgroup the model treats systematically differently, not a fluke. Sort by absolute bias to find
it:

In [ ]:
view = recap.assign(abs_bias=recap["error_mean"].abs()).sort_values("abs_bias", ascending=False)
view[["c", "count", "error_mean", "abs_error_mean", "error_gap", "error_gap_sig",
      "Medu_cat", "sex_F_value", "age_value"]].round(3)

## 6. The heatmap

The same table, read at a glance by colour family instead of by scanning numbers:

- **blue** = cluster size,
- **red** = error (darker = more biased/larger),
- **violet** = sensitive composition,
- p-value columns get darker as they get *more* significant.

Scan for a row that is dark-red in the error column and dark in its `gap sig.` — that is the
disparate cluster — then read across its violet columns to see which group it concentrates.
`error_label` just renames the error column for display.

In [ ]:
plot_cluster_recap_heatmap(recap.copy(), "student_residual", ".", error_label="residual")
Image("student_residual.png")

## 7. Distance matters: Gower on mixed data

The Euclidean baseline in §3 quietly makes a strong assumption: that after one-hot encoding, a
straight-line distance in that space is meaningful. For mixed numeric/categorical data it often is
not, for a concrete geometric reason.

Picture the feature space. Numeric columns like `G1` (grade, 0–20) span a wide numeric range;
each one-hot category column is just 0 or 1. Euclidean distance adds up squared differences across
all axes, so the wide-range numerics dominate and the categorical axes barely register — unless
you standardise, at which point the *opposite* failure appears: `StandardScaler` divides each
column by its standard deviation, and a rare category (a dummy that is 1 for only a handful of
students) has a tiny standard deviation, so scaling *inflates* it into a high-variance axis that
now dominates the distance. Either way the geometry is distorted by an artefact of the encoding,
not by anything real about the students. (This is why §3 kept the dummies out of the scaler — a
partial fix, not a complete one.)

**Gower distance** sidesteps all of this. It compares each feature on its own terms and averages:

- **categorical** features contribute a simple *matching dissimilarity* — 0 if the two students
  share the category, 1 if they do not,
- **numeric** features contribute their difference normalised by the feature's range, so every
  numeric column lands on a comparable 0–1 scale.

No one-hot explosion, no standardisation needed — every feature gets an equal, interpretable vote.
On mixed data Gower can recover subgroups that Euclidean smears together.

Below we cluster with **kmedoids + Gower**. Kmedoids picks actual students as cluster centres
(medoids) rather than averaging coordinates, which is both more robust to outliers and more
natural once distances are non-Euclidean. Since the student columns are already numeric codes, we
just tell Gower which indices to treat as categorical (`sex_F`, `Medu`) — no encoding step — and
standardisation is skipped automatically.

In [ ]:
g_regular = ["G1", "G2", "studytime", "absences"]
g_sensitive = ["sex_F", "Medu", "age"]
g_cols = g_regular + g_sensitive
g_categorical = ["sex_F", "Medu"]                 # treat with matching dissimilarity, not as numbers
cat_idx = [g_cols.index(c) for c in g_categorical]

res_g = cluster(df[g_cols], algorithm="kmedoids", distance="gower",
                categorical_features=cat_idx, n_clusters=4, random_state=11)
print(f"k = {res_g.n_clusters}   silhouette = {res_g.silhouette:.3f}   sizes = {res_g.cluster_sizes}")

In [ ]:
# Compact recap of the Gower partition — median residual per cluster and who it holds.
gdf = df.assign(residual=residual.values, clusters=res_g.labels)
gower_summary = (
    gdf.groupby("clusters")
       .agg(n=("residual", "size"),
            median_residual=("residual", "median"),
            median_age=("age", "median"),
            dominant_Medu=("Medu", lambda s: int(s.mode().iloc[0])),
            pct_female=("sex_F", "mean"))
       .assign(abs_residual=lambda d: d["median_residual"].abs())
       .sort_values("abs_residual", ascending=False)
)
gower_summary.round(3)

The top row is the Gower partition's most-biased cluster — where the model's grades are most
systematically off — and `dominant_Medu` / `median_age` / `pct_female` say for whom. The point of
the exercise is the *comparison*: line this cluster's composition up against the Euclidean recap
from §5. If the two distances concentrate the error in different subgroups, that is a signal the
result is sensitive to the distance choice, and the mixed-type geometry deserves the closer look
Gower gives it. (Whether the subgroup is robust across *many* configurations is the sensitivity
question the `--experiment` sweep in the takeaway is built to answer.)

## 8. Choosing k without guessing: silhouette k-search

Fixing `k=4` was a guess. Instead of committing to one, search a range and let a score pick. The
**silhouette** score rates a clustering from −1 to 1 by asking, for each point, whether it sits
closer to its own cluster than to the nearest other one — high means clusters are both tight
(cohesive) and well-separated. `cluster` sweeps `[n_min, n_max]`, scores each k, and keeps the
best; silhouette is the default scorer.

The CLI spells the same thing `--n_min 2 --n_max 25 --scoring silhouette`. Treat the chosen k as a
*recommendation*, not gospel — if the top few k's score almost the same, the structure is genuinely
ambiguous, which is itself worth knowing.

In [ ]:
res_k = cluster(df[g_cols], algorithm="kmedoids", distance="gower",
                categorical_features=cat_idx, n_min=2, n_max=8, random_state=11)
print(f"selected k = {res_k.n_clusters}   silhouette = {res_k.silhouette:.3f}   sizes = {res_k.cluster_sizes}")

## Takeaway & variations

The whole point is to turn *"the model has RMSE X"* into *"the model is biased by +Y grade points
for **this** subgroup, and the gap is significant"* — a disparity localised to a pocket of the
feature space that a per-attribute average cannot see.

Knobs, and when to reach for each:

- **Bias direction comes for free.** The sign of `error_mean` already separates over- from
  under-prediction — the regression counterpart of the false-positive-vs-false-negative split in
  a classification audit. (Confusion-matrix subsets like `--subset FP_FN` are classification-only;
  in regression the residual's sign carries that information.)
- **Distance:** `distance="gower"` when features are mixed numeric/categorical; plain
  `"euclidean"` (after one-hot) when everything is genuinely numeric.
- **Algorithm:** `kmeans`/`kmedoids` when you want a fixed number of clusters (kmedoids if the
  distance is non-Euclidean or outliers are a concern); `hdbscan` (with `min_samples`,
  `min_datapoints`) to let the data decide how many clusters exist and drop ones too small to
  trust.
- **k:** fix it when you have a reason, or search `n_min`/`n_max` when you do not.

**Batch every combination with the CLI.** Auditing one configuration answers "is there a
disparate subgroup here"; auditing *many* answers the harder question — "is the same subgroup
disparate no matter how I cluster?" `--experiment` mode runs each feature-group × algorithm ×
distance combination and writes an Overview table plus comparison heatmaps:

```bash
python -m c4fairness.main \
  --data_path Data/student_performance.csv \
  --algorithm kmedoids --distance gower --n_min 2 --n_max 25 --scoring silhouette --seed 11 \
  --regular_cols "G1,G2,studytime,absences" \
  --sensitive_cols "sex_F,Medu,age" --continuous_sensitive_cols "age" \
  --categorical_cols "Medu,sex_F" \
  --error_type regression --y_true_col y_true --y_pred_col y_pred \
  --experiment --no_standardize --projection pca
```

Swap `kmedoids --n_min 2 --n_max 25` for `hdbscan --min_samples 20 --min_datapoints 675` to compare
a density-based run against the partition-based one — the same multi-config sweep used to check
whether a finding is real or an artefact of one clustering choice.